In [ ]:
# Clean install
!pip uninstall plotly kaleido -y -q
!pip install plotly==5.24.1 kaleido==0.2.1 python-pptx -q

print("✅ Packages installed. Please Restart Runtime now.")

# Auto restart
import os
os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 13.9 MB/s eta 0:00:00


In [1]:
!pip install plotly==5.24.1 kaleido==0.2.1 python-pptx pandas -q

print("✅ Packages installed successfully!")

✅ Packages installed successfully!


In [2]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

# Load the data
df = pd.read_csv(list(uploaded.keys())[0])

df['Applied Date'] = pd.to_datetime(df['Applied Date'], errors='coerce')
df['Days to Close'] = pd.to_numeric(df['Days to Close'], errors='coerce')

print(f"✅ Successfully loaded {len(df)} candidates!")

Saving SGS_Hiring_Dashboard_SGS_SOP - Sigma Growth Specialists Cohort.csv to SGS_Hiring_Dashboard_SGS_SOP - Sigma Growth Specialists Cohort.csv
✅ Successfully loaded 39 candidates!


In [3]:
import plotly.express as px
import plotly.graph_objects as go

# 1. Funnel
fig_funnel = go.Figure(go.Funnel(
    y=["Applied", "Application Score", "Trial Task", "Interview", "Offer/Hold"],
    x=[len(df), df['App Score'].notna().sum(), df['Trial Task'].notna().sum(),
       df['Interview'].notna().sum(), len(df[df['Status'].isin(['Offer', 'HOLD'])])],
    textinfo="value+percent initial"
))
fig_funnel.update_layout(title="1. Hiring Funnel Overview", height=500)

# 2. Quality by Source
quality_source = pd.crosstab(df['Sources'], df['Quality (G)'], normalize='index') * 100
fig_quality = px.bar(quality_source.reset_index(), x='Sources', y=quality_source.columns,
                     title="2. Quality Tier by Source", barmode='stack')

# 3. Assessment Scores
advanced = df[df['Status'].isin(['Offer', 'HOLD'])]
rejected = df[~df['Status'].isin(['Offer', 'HOLD'])]

score_data = pd.DataFrame({
    'Stage': ['App Score', 'Trial Task', 'Interview'],
    'Advanced/Offer': [advanced['App Score'].mean(), advanced['Trial Task'].mean(), advanced['Interview'].mean()],
    'Rejected': [rejected['App Score'].mean(), None, None]
})

fig_scores = px.bar(score_data.melt(id_vars='Stage'), x='Stage', y='value', color='variable',
                    title="3. Assessment Scores by Outcome", barmode='group')

# 4. Source Performance
source_perf = df.groupby('Sources').agg(
    Candidates=('Candidate Name', 'count'),
    Top_Choice_Rate=('Quality (G)', lambda x: (x == 'Top Choice').mean()*100),
    Offer_Rate=('Status', lambda x: x.isin(['Offer','HOLD']).mean()*100)
).round(1)

fig_source = px.bar(source_perf.reset_index(), x='Sources',
                    y=['Candidates', 'Top_Choice_Rate', 'Offer_Rate'],
                    barmode='group', title="4. Source Performance Comparison")

# 5. Applications Over Time
daily = df.groupby(df['Applied Date'].dt.date).size().reset_index(name='Applications')
fig_time = px.line(daily, x='Applied Date', y='Applications', markers=True,
                   title="5. Applications Over Time")

print("✅ All charts created!")

✅ All charts created!


In [4]:
from pptx import Presentation
from pptx.util import Inches, Pt
from google.colab import files

print("📊 Creating PowerPoint...")

prs = Presentation()
prs.slide_width = Inches(13.33)
prs.slide_height = Inches(7.5)

def add_chart_slide(title, fig, subtitle=""):
    slide = prs.slides.add_slide(prs.slide_layouts[5])
    slide.shapes.title.text = title
    img_path = f"{title[:30].replace(' ', '_')}.png"
    fig.write_image(img_path, width=1200, height=680, scale=2)
    slide.shapes.add_picture(img_path, Inches(0.5), Inches(1.3), width=Inches(12))

    if subtitle:
        txBox = slide.shapes.add_textbox(Inches(0.5), Inches(6.3), Inches(12), Inches(0.8))
        txBox.text_frame.text = subtitle

# Title Slide
slide = prs.slides.add_slide(prs.slide_layouts[0])
slide.shapes.title.text = "SGS Systems & Ops Hiring Dashboard"
slide.placeholders[1].text = "Systems & Infrastructure Role"

# Executive Summary
slide = prs.slides.add_slide(prs.slide_layouts[1])
slide.shapes.title.text = "Executive Summary"
tf = slide.placeholders[1].text_frame
tf.text = f"""Total Candidates: {len(df)}
Avg Days to Close: {round(df['Days to Close'].mean(), 1)} days

Key Insights:
• LinkedIn = Better for volume & Top Choice candidates
• Slack = Much higher conversion & faster closes
• Trial Tasks & Interviews were the strongest validators"""

# Add Charts
add_chart_slide("1. Hiring Funnel Overview", fig_funnel)
add_chart_slide("2. Quality Tier by Source", fig_quality)
add_chart_slide("3. Assessment Scores by Outcome", fig_scores)
add_chart_slide("4. Source Performance", fig_source)
add_chart_slide("5. Applications Over Time", fig_time)

prs.save("SGS_Hiring_Dashboard.pptx")
print("✅ PowerPoint created successfully!")

files.download("SGS_Hiring_Dashboard.pptx")

📊 Creating PowerPoint...
✅ PowerPoint created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
from google.colab import files


# Download the notebook itself
# files.download("SGS_Hiring_Dashboard.ipynb")   # If it asks to save first, click File → Save